# 02 — Data Collection: Web Scraping (Wikipedia)

Parses the "List of Falcon 9 and Falcon Heavy launches" Wikipedia page with
BeautifulSoup to build an independent launch-record table for cross-checking
against the API dataset.

**Requires internet access to run.**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
resp = requests.get(URL)
soup = BeautifulSoup(resp.content, "html.parser")
print(resp.status_code, soup.title.text)


### Locate the launch tables

In [ ]:
tables = soup.find_all("table", {"class": "wikitable plainrowheaders collapsible"})
print(f"Found {len(tables)} launch tables")


### Parse rows into records

Handles footnote references (`[1]`, `[note 2]`) and merged cells by
parsing defensively and skipping malformed rows rather than crashing.

In [ ]:
def clean_text(cell):
    text = cell.get_text(strip=True)
    text = re.sub(r"\[.*?\]", "", text)  # strip footnote refs like [1]
    return text.strip()

records = []
for table in tables:
    rows = table.find_all("tr")
    for row in rows:
        cells = row.find_all(["th", "td"])
        if len(cells) < 6:
            continue
        try:
            record = {
                "FlightNumber": clean_text(cells[0]),
                "Date": clean_text(cells[1]) if len(cells) > 1 else None,
                "BoosterVersion": clean_text(cells[2]) if len(cells) > 2 else None,
                "LaunchSite": clean_text(cells[3]) if len(cells) > 3 else None,
                "Payload": clean_text(cells[4]) if len(cells) > 4 else None,
                "Orbit": clean_text(cells[6]) if len(cells) > 6 else None,
                "Outcome": clean_text(cells[-1]),
            }
            records.append(record)
        except Exception:
            continue

df_scraped = pd.DataFrame(records)
print(df_scraped.shape)
df_scraped.head()


### Save for cross-checking against the API dataset

In [ ]:
df_scraped.to_csv("../falcon9_scraped_raw.csv", index=False)
